# 找谁来帮忙：生成语音（第二个办法）
1. 点上方菜单 **Runtime → Run all**
2. 如果出现警告，点 **Run anyway**
3. 等 3-6 分钟，最后浏览器会下载 `voice.zip`
4. 把 `voice.zip` 发回给 Claude

In [ ]:
# 第 1 步：安装语音工具（约 1-2 分钟）
!pip -q install kokoro soundfile "misaki[zh]"
!apt-get -qq install -y espeak-ng ffmpeg > /dev/null

# 第 2 步：把每一句话变成声音（约 3-6 分钟）
import json, os, zipfile, subprocess
import numpy as np, soundfile as sf
from kokoro import KPipeline
from google.colab import files

VOICE = "zf_xiaoxiao"   # 中文女声。也可以试试 zf_xiaoyi、zf_xiaobei、zf_xiaoni
SPEED = 0.9             # 说慢一点，一年级学生比较容易听懂

SENTENCES = [
 "面对紧急状况，不要慌。遇到紧急状况，可以大声向路人求助，拨打九九九，或者向成人求助。",
 "房子着火了！浓烟冒出来。",
 "有人在水里喊“救命”。",
 "同学跌倒了，流很多血。",
 "在商场里和妈妈走散了。我迷路了。",
 "今天有雾霾。我在学校，一直咳嗽，喘不过气。",
 "我想吃糖果，也想要新玩具。",
 "玩具车的电池没电了。",
 "公路上发生车祸，有人受伤。",
 "我迷路了。",
 "我很怕，膝盖流血了。",
 "弟弟跌倒了，大哭“妈妈！”",
 "吹风筒坏了，冒出烟来。",
 "陌生人说：“我有糖果和玩具，跟我来拿！”",
 "雾霾天，下课了。",
 "做得好",
 "太棒了！遇到紧急状况，不要慌，大声求助，拨打九九九，或者向成人求助。",
 "小朋友，你怎么了？",
 "别怕。我会用广播帮你找妈妈。你知道妈妈的电话号码吗？",
 "小华！妈妈在这里！",
 "你做得很好！迷路了要冷静，向保安或职员求助。",
 "小朋友，我有糖果和新玩具，跟我到车上拿吧！",
 "来嘛，很近而已，我拉你过去！",
 "怎么了？",
 "你别怕，老师会请保安和你的爸爸妈妈来。",
 "你做得很对！不可以接受陌生人的糖果和玩具，要大声求助。",
 "你怎么了？",
 "别怕，老师带你去医务室，也会通知你的爸爸妈妈。如果很严重，大人会拨打 九九九。",
 "你做得很好！雾霾天少出去，不舒服要马上告诉大人。",
 "我在这里！别怕。你站着别动，妈妈拿手电筒来。",
 "妈妈在这里。我们用手电筒，等电来。",
 "你做得很好！停电时不要慌，留在原地，找大人帮忙，不要自己玩火。",
 "我是送快递的，请开门！",
 "小朋友，我有糖果和玩具，快开门！",
 "爸爸马上回来。你不要开门。如果有人撞门，请叫邻居，大人会拨 九九九。",
 "你做得很对！爸爸妈妈不在时，不开门给陌生人，要告诉大人。",
 "您好，这里是 九九九。请问发生什么事？",
 "请问你在哪里？",
 "好的，我们马上来。你叫什么名字？",
 "知道了。请站在安全的地方，等大人来。",
 "你做得很好！打 九九九 要说清楚：发生什么事、在哪里、你的名字。"
]

pipe = KPipeline(lang_code="z")
os.makedirs("voice", exist_ok=True)
for i, t in enumerate(SENTENCES):
    parts = []
    for _, _, audio in pipe(t, voice=VOICE, speed=SPEED):
        parts.append(audio.numpy() if hasattr(audio, "numpy") else np.asarray(audio))
    sf.write("tmp.wav", np.concatenate(parts), 24000)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "tmp.wav", "-ac", "1", "-b:a", "56k", f"voice/{i:03d}.mp3"], check=True)
    print("完成", i + 1, "/", len(SENTENCES), t[:20])
json.dump(SENTENCES, open("voice/index.json", "w", encoding="utf8"), ensure_ascii=False)

# 第 3 步：打包下载
with zipfile.ZipFile("voice.zip", "w") as z:
    for f in sorted(os.listdir("voice")):
        z.write("voice/" + f)
files.download("voice.zip")
